# CymbalGoal — Stage 3: Schema Build

**Input:** the pinned Transfermarkt snapshot from Stage 1
**Output:** eight schema-shaped, FK-clean Parquet tables + `schema.sql` + a manifest stub

This notebook takes the raw snapshot and produces the exact tables the AlloyDB loader will
import. It does **not** generate profile text or embeddings — that's Stage 4/5, which needs
Vertex AI and runs separately.

### What changed since the Stage 2 run

| # | Fix | Why |
| :-- | :-- | :-- |
| 1 | **Competition discovery raises instead of silently narrowing** | The Stage 2 run matched no UEFA competitions and quietly fell through to Big 5 only. Every downstream number was wrong and nothing said so. Closes **D-15** |
| 2 | **Orphan audit is re-run post-scope** | Stage 2 measured orphans on full tables. Scoped rates are what the DDL is designed against. Closes **D-16** |
| 3 | **Orphaned FKs are NULLed before export** | `transfers.from_club_id` is 62.6% orphaned against a 796-row `clubs` table. Unhandled, the `COPY` dies inside Terraform at Start Lab. Implements **S-24** |
| 4 | **Player scope = option A** | Ever appeared in a scope competition. Lionel Messi's current club is Inter Miami, so "current squads" would drop him and take Lab 1's headline query with him. Implements **S-23** |

> **Run order matters.** Section 2 must succeed before anything else runs. If it raises,
> stop and fix the competition scope — don't comment out the check.

---
## 1. Configuration

In [ ]:
# Colab Enterprise images vary; make the two hard dependencies explicit.
import importlib, subprocess, sys
for mod, pkg in [("pyarrow", "pyarrow"), ("pandas", "pandas")]:
    if importlib.util.find_spec(mod) is None:
        print(f"installing {pkg}…")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
print("dependencies ok")

In [ ]:
import hashlib, io, json, os, re, zipfile
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 100)

# --- Paths -----------------------------------------------------------------
BASE     = Path("/content/cymbalgoal")
RAW      = BASE / "raw"          # the pinned snapshot zip
PARQUET  = BASE / "parquet"      # Stage 3 output, typed
ARTIFACT = BASE / "artifacts"    # schema.sql, manifest.json, reports
for d in (BASE, RAW, PARQUET, ARTIFACT):
    d.mkdir(parents=True, exist_ok=True)

# --- The pin (from the Stage 1 / Stage 2 manifest) -------------------------
SNAPSHOT_URL    = "https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/transfermarkt-datasets.zip"
SNAPSHOT_DATE   = "2026-08-14"
SNAPSHOT_SHA256 = "3e6742a9c5002b202eeb8803a123bbc2d9c13ae582eb45ffb6f587843693fffa"
SNAPSHOT_BYTES  = 228_537_407
ZIP_PATH        = RAW / "transfermarkt-datasets.zip"

# --- Scope (S-21, Part 4 of the decisions log) -----------------------------
BIG5 = ["GB1", "ES1", "IT1", "L1", "FR1"]

# UEFA ids are DISCOVERED, never hardcoded — upstream has changed them historically.
# Section 2 fills this in and raises if it finds nothing.
UEFA_COMPS = []
SCOPE_COMPS = []

# Set True only if you have deliberately decided CymbalGoal is Big-5-only.
# Leaving this False is what makes a missing-UEFA run fail loudly instead of silently.
ALLOW_BIG5_ONLY = False

# --- Embedding contract (S-18) ---------------------------------------------
EMBEDDING_MODEL = "gemini-embedding-001"
EMBEDDING_DIMS  = 3072      # LOCKED. See P-25 — not a free parameter.
VECTOR_SIGFIGS  = 6         # 2.15x smaller CSV, no measurable quality cost

# --- Destination (S-19) ----------------------------------------------------
GCS_PREFIX = "gs://class-demo/alloydb-labs/cymbalgoal"

print(f"snapshot  {SNAPSHOT_DATE}  {SNAPSHOT_SHA256[:16]}…")
print(f"scope     Big 5 = {BIG5}   + UEFA (discovered in section 2)")
print(f"embedding {EMBEDDING_MODEL} @ {EMBEDDING_DIMS}d, {VECTOR_SIGFIGS} sig digits")
print(f"dest      {GCS_PREFIX}")


### 1.1 Fast path — resume from the Parquet checkpoint

**Run this before 1.2.** If a previous run already produced the Parquet tables, everything from
here to section 9 is redundant: the relational extract is deterministic and already on disk.

This matters for two reasons. Re-running the whole pipeline means re-downloading 228 MB and
re-deriving eight tables just to export CSVs that depend on none of it. And if the snapshot host
is unreachable from your network — a 403, a proxy, a bad day at Cloudflare — the fast path lets
Stage 6a run anyway.

**If this cell says RESUMED, skip straight to section 9a.**


In [ ]:

RESUMED = False
_pq = sorted(PARQUET.glob("*.parquet"))
_mf = ARTIFACT / "manifest.json"
_sql = ARTIFACT / "schema.sql"

if len(_pq) == 8 and _mf.exists() and _sql.exists():
    T = {p.stem: pd.read_parquet(p) for p in _pq}
    DDL = _sql.read_text()
    _m = json.loads(_mf.read_text())

    # Restore the values later cells expect, so the manifest rewrite stays faithful.
    LOAD_ORDER      = _m["load_order"]
    SCOPE_COMPS     = _m["scope"]["competitions"]
    BIG5            = _m["scope"]["big5"]
    UEFA_COMPS      = _m["scope"]["uefa_discovered"]
    repairs         = _m.get("fk_repairs", {})
    EVENT_PK_SOURCE = _m.get("event_pk_source", "unknown")
    written         = {p.stem: p.stat().st_size for p in _pq}

    if _m["snapshot"]["sha256"] != SNAPSHOT_SHA256:
        raise RuntimeError(
            "The checkpoint on disk was built from a DIFFERENT snapshot than this notebook "
            f"is pinned to.\n  checkpoint: {_m['snapshot']['sha256']}\n  notebook:   {SNAPSHOT_SHA256}\n"
            "Delete the parquet directory and run the full pipeline, or fix the pin."
        )

    print("✅ RESUMED from the Parquet checkpoint — skip to section 9a\n")
    print(f"   snapshot {_m['snapshot']['date']}  {_m['snapshot']['sha256'][:16]}…")
    print(f"   scope    {', '.join(SCOPE_COMPS)}")
    print(f"   event PK {EVENT_PK_SOURCE}")
    print(f"   schema.sql {len(DDL):,} bytes\n")
    for t in LOAD_ORDER:
        print(f"   {t:<20}{len(T[t]):>10,} rows  {len(T[t].columns):>3} cols")
    RESUMED = True
else:
    missing = []
    if len(_pq) != 8: missing.append(f"parquet ({len(_pq)}/8 files)")
    if not _mf.exists(): missing.append("manifest.json")
    if not _sql.exists(): missing.append("schema.sql")
    print(f"no usable checkpoint ({', '.join(missing)}) — run the full pipeline from 1.2")


### 1.2 Load the pinned snapshot

Reuses the local zip if its checksum matches the pin. Only re-downloads if it's missing or
altered — and a checksum mismatch is a hard stop, because upstream refreshes weekly and a
silently-updated snapshot would invalidate every row count the labs assert against.

In [ ]:
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def _stream_download(url, dest, chunk=1 << 20):
    """Cloudflare R2 rejects the default Python-urllib User-Agent with 403.
    requests plus an explicit browser UA is the path the Stage 2 notebook proved works."""
    import requests, time
    headers = {
        "User-Agent": ("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                       "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"),
        "Accept": "*/*",
    }
    t0 = time.time()
    with requests.get(url, stream=True, timeout=300, headers=headers) as r:
        if r.status_code == 403:
            raise RuntimeError(
                f"403 Forbidden from {url}\n"
                "The bucket is public, so this is almost always User-Agent filtering or a\n"
                "network egress policy — not a credentials problem. Two things to try:\n"
                "  1. curl -A 'Mozilla/5.0' -o snapshot.zip '<url>'   (then re-run: the\n"
                "     checksum check will pick the local file up)\n"
                "  2. Per-table fallback, same host:\n"
                "     https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/players.csv.gz\n"
                "If neither works from this network, use the Parquet fast path in 1.1 —\n"
                "Stage 6a does not need the raw snapshot at all."
            )
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        got = 0
        with open(dest, "wb") as f:
            for blk in r.iter_content(chunk):
                f.write(blk); got += len(blk)
                if total:
                    print(f"\r  {got/1e6:,.0f} / {total/1e6:,.0f} MB", end="")
    print(f"\n  {dest.stat().st_size/1e6:,.1f} MB in {time.time()-t0:,.0f}s")


def ensure_snapshot():
    if ZIP_PATH.exists():
        actual = sha256_file(ZIP_PATH)
        if actual == SNAPSHOT_SHA256:
            print(f"✅ pinned snapshot present and verified ({ZIP_PATH.stat().st_size:,} bytes)")
            return
        print(f"⚠️  local zip checksum mismatch\n     expected {SNAPSHOT_SHA256}\n     actual   {actual}\n   re-downloading…")

    print(f"downloading {SNAPSHOT_URL} …")
    _stream_download(SNAPSHOT_URL, ZIP_PATH)

    actual = sha256_file(ZIP_PATH)
    size = ZIP_PATH.stat().st_size
    if actual != SNAPSHOT_SHA256:
        raise RuntimeError(
            "SNAPSHOT DRIFT — the upstream file no longer matches the pin.\n"
            f"  expected sha256 {SNAPSHOT_SHA256} ({SNAPSHOT_BYTES:,} bytes)\n"
            f"  actual   sha256 {actual} ({size:,} bytes)\n"
            "Upstream refreshes weekly. Either restore the pinned copy, or deliberately "
            "re-pin and regenerate EVERY downstream artifact and lab verification value."
        )
    print(f"✅ downloaded and verified ({size:,} bytes)")


if globals().get("RESUMED"):
    print("skipped — resumed from the Parquet checkpoint (1.1)")
else:
    ensure_snapshot()


In [ ]:
TABLE_FILES = {}   # logical name -> name inside the zip

with zipfile.ZipFile(ZIP_PATH) as z:
    for info in z.infolist():
        if info.filename.endswith(".csv.gz") or info.filename.endswith(".csv"):
            stem = Path(info.filename).name
            stem = stem.replace(".csv.gz", "").replace(".csv", "")
            TABLE_FILES[stem] = info.filename

print(f"{len(TABLE_FILES)} tables in the archive:")
for k in sorted(TABLE_FILES):
    print(f"   {k}")

In [ ]:
WANTED = ["competitions", "clubs", "players", "games",
          "appearances", "game_events", "player_valuations", "transfers"]

src = {}   # logical name -> raw DataFrame, exactly as upstream ships it

with zipfile.ZipFile(ZIP_PATH) as z:
    for name in WANTED:
        if name not in TABLE_FILES:
            raise KeyError(f"{name!r} not found in the snapshot. Present: {sorted(TABLE_FILES)}")
        with z.open(TABLE_FILES[name]) as fh:
            buf = io.BytesIO(fh.read())
        compression = "gzip" if TABLE_FILES[name].endswith(".gz") else None
        src[name] = pd.read_csv(buf, compression=compression, low_memory=False)
        print(f"  {name:<20} {len(src[name]):>10,} rows  {len(src[name].columns):>3} cols")

print(f"\nloaded {len(src)}/{len(WANTED)} tables "
      f"(game_lineups, club_games, countries, national_teams intentionally skipped — S-20)")

---
## 2. Competition scope — the check that failed silently last time

The Stage 2 run searched competition names for UEFA competitions, found nothing, printed a
warning, and **carried on with Big 5 only**. Every scoped count downstream was wrong, and
the only evidence was two lines of log buried in a long report.

This version does three things differently:

1. Searches **id, name, and sub_type** — not just name — because upstream slugs are
   inconsistent across competition types.
2. Prints the full competition inventory unconditionally, so the scope is auditable rather
   than inferred from downstream row counts.
3. **Raises** if no UEFA club competition is found. A scope this central should never be
   decided by a fallthrough.

In [ ]:
comp = src["competitions"]

print(f"competitions table: {len(comp):,} rows, columns: {list(comp.columns)}\n")

# Full inventory — the thing whose absence hid the bug last time.
inv_cols = [c for c in ["competition_id", "name", "type", "sub_type", "country_name"]
            if c in comp.columns]
print("FULL COMPETITION INVENTORY")
print("=" * 100)
for t, grp in comp.groupby(comp["type"] if "type" in comp.columns else pd.Series("?", index=comp.index)):
    print(f"\n--- {t}  ({len(grp)}) ---")
    print(grp[inv_cols].to_string(index=False))

In [ ]:
# --- Big 5: verify each code is really present -----------------------------
present = set(comp["competition_id"])
missing_big5 = [c for c in BIG5 if c not in present]
if missing_big5:
    raise ValueError(f"Big 5 codes missing from the snapshot: {missing_big5}")

print("Big 5 verified:")
for c in BIG5:
    row = comp.loc[comp.competition_id == c].iloc[0]
    print(f"   {c:<5} {row.get('name','?'):<40} {row.get('country_name','?')}")

# --- UEFA: search id, name, AND sub_type -----------------------------------
# Cast wide, then decide deliberately. The Europa League is typed 'other' upstream,
# not 'international_cup', which is exactly why a narrow search missed it last time.
UEFA_PATTERNS = [
    r"champions[-_ ]?league",
    r"europa[-_ ]?league",
    r"conference[-_ ]?league",     # UECL slug has no "europa" in it
    r"uefa[-_ ]?conference",
    r"uefa[-_ ]?cup",
]
EXCLUDE_PATTERNS = [r"super[-_ ]?cup", r"women", r"youth", r"u1[789]", r"qualif"]

# Of what we discover, these enter scope. UECL is deliberately absent: it only exists
# from 2021 and would skew season coverage. Anything found but not matched here is
# reported loudly rather than silently dropped — a regex should not decide scope.
SCOPE_SUBTYPE_PATTERNS = [r"champions[-_ ]?league", r"europa[-_ ]?league"]

def _hay(row):
    parts = [str(row.get(c, "")) for c in ("competition_id", "name", "sub_type", "type")]
    return " ".join(parts).lower()

hits = []
for _, row in comp.iterrows():
    h = _hay(row)
    if any(re.search(p, h) for p in UEFA_PATTERNS) and not any(re.search(p, h) for p in EXCLUDE_PATTERNS):
        hits.append(row)

UEFA_COMPS, UEFA_FOUND_NOT_SCOPED = [], []
for r in hits:
    h = _hay(r)
    (UEFA_COMPS if any(re.search(p, h) for p in SCOPE_SUBTYPE_PATTERNS)
     else UEFA_FOUND_NOT_SCOPED).append(r)

print("\nUEFA competitions IN SCOPE:")
for r in UEFA_COMPS:
    print(f"   {r['competition_id']:<8} {r.get('name','?'):<45} type={r.get('type','?')}")
if not UEFA_COMPS:
    print("   (none matched)")

if UEFA_FOUND_NOT_SCOPED:
    print("\n⚠️  FOUND BUT NOT IN SCOPE — confirm this is what you want:")
    for r in UEFA_FOUND_NOT_SCOPED:
        print(f"   {r['competition_id']:<8} {r.get('name','?'):<45} type={r.get('type','?')}")
    print("   To include one, add its pattern to SCOPE_SUBTYPE_PATTERNS and re-run.")

UEFA_COMPS = [r["competition_id"] for r in UEFA_COMPS]

In [ ]:
SCOPE_COMPS = BIG5 + UEFA_COMPS

if not UEFA_COMPS:
    intl = comp.loc[comp.get("type", pd.Series(dtype=str)) == "international_cup"] \
           if "type" in comp.columns else comp.iloc[0:0]
    msg = (
        "\n"
        "=" * 78 + "\n"
        "D-15 — NO UEFA CLUB COMPETITION FOUND. STOPPING.\n"
        + "=" * 78 + "\n"
        "The settled scope (decisions log, Part 4) is Big 5 + UCL/UEL. Nothing matched the\n"
        "UEFA patterns, so continuing would silently produce a Big-5-only dataset — exactly\n"
        "the failure this check exists to prevent.\n\n"
        "Look at the inventory printed above. The competitions typed 'international_cup' are:\n"
        f"{intl[[c for c in ['competition_id','name','sub_type'] if c in intl.columns]].to_string(index=False) if len(intl) else '   (none)'}\n\n"
        "Then either:\n"
        "  (a) add the correct id patterns to UEFA_PATTERNS above and re-run, or\n"
        "  (b) if the snapshot genuinely has no UCL/UEL, set ALLOW_BIG5_ONLY = True in\n"
        "      section 1, re-run, and AMEND Part 4 of the decisions log to match reality.\n"
        + "=" * 78
    )
    if not ALLOW_BIG5_ONLY:
        raise RuntimeError(msg)
    print(msg)
    print("\n⚠️  ALLOW_BIG5_ONLY is set — proceeding Big-5-only, deliberately.")

print(f"\nSCOPE_COMPS ({len(SCOPE_COMPS)}): {SCOPE_COMPS}")

---
## 3. Column reconnaissance

Every rename below is written against upstream column names, and upstream renames columns
between refreshes. Rather than crash three cells later with a `KeyError`, this prints what
actually exists so the mapping in section 5 can be corrected against reality.

In [ ]:
for name in WANTED:
    print(f"--- {name} ({len(src[name]):,} rows) ---")
    print("   " + ", ".join(src[name].columns))
    print()

---
## 4. Build the scope sets

**S-21:** dimensions load whole, facts get scoped.
**S-23:** players = option A, anyone who ever appeared in a scope competition.

Option A rather than "current squads" is not a size preference. Lionel Messi's
`current_club_id` is Inter Miami, so a current-squad rule drops him from the corpus — and
Lab 1's headline query is a fan searching *"Messi"*.

In [ ]:
games_raw = src["games"]
apps_raw  = src["appearances"]

scope_game_ids = set(games_raw.loc[games_raw.competition_id.isin(SCOPE_COMPS), "game_id"])
print(f"scope games:   {len(scope_game_ids):,} of {len(games_raw):,} "
      f"({len(scope_game_ids)/len(games_raw):.1%})")

# S-23 / option A — ever appeared in a scope competition, any season.
apps_in_scope = apps_raw[apps_raw.game_id.isin(scope_game_ids)]
scope_player_ids = set(apps_in_scope.player_id.unique())
print(f"scope players: {len(scope_player_ids):,} of {len(src['players']):,} "
      f"({len(scope_player_ids)/len(src['players']):.1%})")

# --- Canary: the query Lab 1 is built around --------------------------------
players_raw = src["players"]
name_col = "name" if "name" in players_raw.columns else "player_name"
messi = players_raw[players_raw[name_col].str.contains("messi", case=False, na=False)]

print("\nLab 1 canary — 'messi' matches, and whether each survives scoping:")
for _, r in messi.iterrows():
    inn = "IN SCOPE " if r.player_id in scope_player_ids else "   ---   "
    print(f"   {inn} id={r.player_id:<9} {r[name_col]}")

lionel = messi[messi[name_col].str.strip().str.lower() == "lionel messi"]
if len(lionel) == 0:
    raise RuntimeError("Lionel Messi is not in the players table at all — check the snapshot.")
if not set(lionel.player_id).issubset(scope_player_ids):
    raise RuntimeError(
        "CANARY FAILED — Lionel Messi did not survive scoping.\n"
        "Lab 1's headline query ('a fan searches Messi') has no correct answer in this corpus.\n"
        "Check the player scope rule before continuing."
    )
print("\n✅ canary passes — Lionel Messi is in the corpus, with decoys for BM25 to rank against")

---
## 5. Transform to the schema

Four rules, all in service of Lab 2 pointing an LLM at this schema and getting correct SQL:

1. **No column named after a SQL keyword, type, or function.** `date` → `game_date`,
   `type` → `event_type`, `position` → `main_position`.
2. **One name per concept, entity-qualified.** Three tables ship a column called `name`.
3. **Drop denormalized name columns where an FK exists.** `appearances.player_name` lets an
   LLM answer "top scorers" without ever joining `players` — so the lab never demonstrates
   the join it's meant to teach, using a name that's gone stale.
4. **Parse formatted strings into real types.** `net_transfer_record` is `'+€5.90m'`. An LLM
   will `SUM` it.

In [ ]:
def parse_money_eur(s):
    """'+€5.90m' -> 5900000 ; '€-25.00m' -> -25000000 ; '' / '+-0' -> 0"""
    if pd.isna(s):
        return None
    t = str(s).strip().replace("€", "").replace("£", "").replace("+", "").replace(",", "")
    if t in ("", "-", "0", "-0", "+-0"):
        return 0
    mult = 1
    if t[-1:].lower() == "m":
        mult, t = 1_000_000, t[:-1]
    elif t[-1:].lower() in ("k", "th"):
        mult, t = 1_000, t[:-1]
    elif t[-2:].lower() == "bn":
        mult, t = 1_000_000_000, t[:-2]
    try:
        return int(round(float(t) * mult))
    except ValueError:
        return None


def season_int(s):
    """clubs.last_season ships as a date, players.last_season as an int. Normalize to int."""
    if pd.isna(s):
        return None
    t = str(s)
    m = re.match(r"^(\d{4})", t)
    return int(m.group(1)) if m else None


def build(name, rename, keep_extra=(), derive=None):
    """Rename what exists, report what doesn't, return only schema columns."""
    df = src[name].copy()
    have, missing = {}, []
    for tgt, s in rename.items():
        if s in df.columns:
            have[s] = tgt
        else:
            missing.append(f"{tgt} <- {s}")
    out = df.rename(columns=have)
    if derive:
        out = derive(out)
    cols = [c for c in list(rename.keys()) + list(keep_extra) if c in out.columns]
    out = out[cols]
    if missing:
        print(f"  ⚠️  {name}: {len(missing)} source column(s) not found — "
              f"emitted as NULL: {', '.join(missing)}")
        for m in missing:
            tgt = m.split(" <- ")[0]
            if tgt not in out.columns:
                out[tgt] = pd.NA
        out = out[[c for c in list(rename.keys()) + list(keep_extra) if c in out.columns]]
    return out


def stable_id(df, cols, bits=63):
    """Deterministic surrogate key from a natural key. Same input -> same id, every run."""
    key = df[cols].astype(str).agg("\x1f".join, axis=1)
    occ = key.groupby(key).cumcount().astype(str)
    full = key + "\x1f#" + occ
    mask = (1 << bits) - 1
    return full.map(lambda s: int.from_bytes(
        hashlib.blake2b(s.encode(), digest_size=8).digest(), "big") & mask)


print("transform helpers ready")

In [ ]:
T = {}   # schema-shaped tables

# --- competitions (whole) --------------------------------------------------
def _comps_derive(d):
    """Upstream types the Europa League 'other', not 'international_cup'. A filter on
    competition_type therefore silently misses it — precisely the kind of query Lab 2's
    LLM will write. Normalize UEFA club competitions to the type they plainly are."""
    if {"competition_type", "competition_sub_type"} <= set(d.columns):
        m = (d["competition_sub_type"].astype(str)
               .str.contains(r"uefa_(?:champions|europa|conference)", case=False, na=False)
             & (d["competition_type"] != "international_cup"))
        if m.any():
            print(f"  \u2139\ufe0f  competition_type normalized to 'international_cup' for "
                  f"{d.loc[m, 'competition_id'].tolist()} "
                  f"(upstream said {sorted(d.loc[m, 'competition_type'].astype(str).unique())})")
            print("      This intentionally includes qualifying rounds. Upstream is "
                  "self-inconsistent - it types CLQ as international_cup but ELQ as "
                  "'other' - so uniform beats faithful here.")
        d.loc[m, "competition_type"] = "international_cup"
    return d


T["competitions"] = build("competitions", {
    "competition_id":       "competition_id",
    "competition_code":     "competition_code",
    "competition_name":     "name",
    "competition_type":     "type",
    "competition_sub_type": "sub_type",
    "country_name":         "country_name",
    "domestic_league_code": "domestic_league_code",
    "confederation":        "confederation",
    "total_clubs":          "total_clubs",
    "url":                  "url",
}, derive=_comps_derive)

# --- clubs (whole) ---------------------------------------------------------
def _clubs_derive(d):
    if "net_transfer_record" in d.columns:
        d["net_transfer_record_eur"] = d["net_transfer_record"].map(parse_money_eur)
    if "last_season" in d.columns:
        d["last_season"] = d["last_season"].map(season_int)
    return d

T["clubs"] = build("clubs", {
    "club_id":                 "club_id",
    "club_code":               "club_code",
    "club_name":               "name",
    "domestic_competition_id": "domestic_competition_id",
    "squad_size":              "squad_size",
    "average_age":             "average_age",
    "foreigners_number":       "foreigners_number",
    "foreigners_percentage":   "foreigners_percentage",
    "national_team_players":   "national_team_players",
    "stadium_name":            "stadium_name",
    "stadium_seats":           "stadium_seats",
    "coach_name":              "coach_name",
    "last_season":             "last_season",
    "url":                     "url",
}, keep_extra=("net_transfer_record_eur",), derive=_clubs_derive)

# clubs.total_market_value is 100% NULL in this snapshot (P-30) — deliberately not loaded.

# --- players (scoped, option A) --------------------------------------------
def _players_derive(d):
    if "last_season" in d.columns:
        d["last_season"] = d["last_season"].map(season_int)
    return d

_players = build("players", {
    "player_id":                   "player_id",
    "player_name":                 "name",
    "first_name":                  "first_name",
    "last_name":                   "last_name",
    "current_club_id":             "current_club_id",
    "country_of_citizenship":      "country_of_citizenship",
    "country_of_birth":            "country_of_birth",
    "city_of_birth":               "city_of_birth",
    "date_of_birth":               "date_of_birth",
    "main_position":               "position",
    "detailed_position":           "sub_position",
    "foot":                        "foot",
    "height_in_cm":                "height_in_cm",
    "market_value_in_eur":         "market_value_in_eur",
    "highest_market_value_in_eur": "highest_market_value_in_eur",
    "contract_expiration_date":    "contract_expiration_date",
    "agent_name":                  "agent_name",
    "international_caps":          "international_caps",
    "international_goals":         "international_goals",
    "last_season":                 "last_season",
    "url":                         "url",
}, derive=_players_derive)
T["players"] = _players[_players.player_id.isin(scope_player_ids)].copy()

# Profile columns are populated by Stage 4/5 and loaded in a second pass.
T["players"]["profile_text"] = pd.NA
T["clubs"]["profile_text"]   = pd.NA

for k, v in T.items():
    print(f"  {k:<20} {len(v):>10,} rows  {len(v.columns):>3} cols")

In [ ]:
# --- games (scoped) --------------------------------------------------------
_games = build("games", {
    "game_id":                "game_id",
    "competition_id":         "competition_id",
    "season":                 "season",
    "round_label":            "round",
    "game_date":              "date",
    "home_club_id":           "home_club_id",
    "away_club_id":           "away_club_id",
    "home_club_goals":        "home_club_goals",
    "away_club_goals":        "away_club_goals",
    "home_club_position":     "home_club_position",
    "away_club_position":     "away_club_position",
    "home_club_manager_name": "home_club_manager_name",
    "away_club_manager_name": "away_club_manager_name",
    "home_club_formation":    "home_club_formation",
    "away_club_formation":    "away_club_formation",
    "aggregate":              "aggregate",
    "stadium":                "stadium",
    "attendance":             "attendance",
    "referee":                "referee",
    "url":                    "url",
})
T["games"] = _games[_games.game_id.isin(scope_game_ids)].copy()

# --- appearances (scoped by game) ------------------------------------------
_apps = build("appearances", {
    "appearance_id":   "appearance_id",
    "game_id":         "game_id",
    "player_id":       "player_id",
    "player_club_id":  "player_club_id",
    "competition_id":  "competition_id",
    "appearance_date": "date",
    "yellow_cards":    "yellow_cards",
    "red_cards":       "red_cards",
    "goals":           "goals",
    "assists":         "assists",
    "minutes_played":  "minutes_played",
})
T["appearances"] = _apps[_apps.game_id.isin(scope_game_ids)].copy()

# --- game_events (scoped by game) ------------------------------------------
_ev = build("game_events", {
    "game_event_id":    "game_event_id",
    "game_id":          "game_id",
    "player_id":        "player_id",
    "club_id":          "club_id",
    "event_date":       "date",
    "event_type":       "type",
    "minute":           "minute",
    "description":      "description",
    "player_in_id":     "player_in_id",
    "player_assist_id": "player_assist_id",
})
_ev = _ev[_ev.game_id.isin(scope_game_ids)].copy()
_ev = _ev.sort_values(["game_id", "minute", "event_type", "player_id"], kind="mergesort")

# Prefer upstream's natural key when it is actually usable. Falling back to a generated
# surrogate is fine, but a natural key lets lab verification steps cite real event ids.
_nat = _ev["game_event_id"] if "game_event_id" in _ev.columns else None
if _nat is not None and _nat.notna().all() and not _nat.duplicated().any():
    EVENT_PK_SOURCE, EVENT_PK_TYPE = "upstream natural key", "TEXT"
    _ev["game_event_id"] = _nat.astype(str)
    print(f"  \u2705 game_events PK: upstream game_event_id ({_nat.nunique():,} distinct)")
else:
    _n_null = int(_nat.isna().sum()) if _nat is not None else -1
    _n_dup = int(_nat.duplicated().sum()) if _nat is not None else -1
    EVENT_PK_SOURCE, EVENT_PK_TYPE = "generated surrogate (BLAKE2b)", "BIGINT"
    print(f"  \u26a0\ufe0f  upstream game_event_id unusable "
          f"(nulls={_n_null:,}, dupes={_n_dup:,}) — generating a deterministic surrogate")
    _ev["game_event_id"] = stable_id(
        _ev, ["game_id", "minute", "event_type", "player_id", "description"])

T["game_events"] = _ev[["game_event_id"] + [c for c in _ev.columns if c != "game_event_id"]]

# --- player_valuations (scoped by player) ----------------------------------
_val = build("player_valuations", {
    "player_id":           "player_id",
    "valuation_date":      "date",
    "market_value_in_eur": "market_value_in_eur",
    "current_club_id":     "current_club_id",
})
_val = _val[_val.player_id.isin(scope_player_ids)].copy()
before = len(_val)
_val = _val.drop_duplicates(subset=["player_id", "valuation_date"], keep="last")
if before != len(_val):
    print(f"  ℹ️  player_valuations: dropped {before-len(_val):,} duplicate (player_id, valuation_date) rows")
T["player_valuations"] = _val

# --- transfers (scoped by player) ------------------------------------------
_tr = build("transfers", {
    "player_id":           "player_id",
    "transfer_date":       "transfer_date",
    "transfer_season":     "transfer_season",
    "from_club_id":        "from_club_id",
    "from_club_name":      "from_club_name",
    "to_club_id":          "to_club_id",
    "to_club_name":        "to_club_name",
    "transfer_fee":        "transfer_fee",
    "market_value_in_eur": "market_value_in_eur",
})
_tr = _tr[_tr.player_id.isin(scope_player_ids)].copy()
_tr = _tr.sort_values(["player_id", "transfer_date", "from_club_id", "to_club_id"], kind="mergesort")
_tr["transfer_id"] = stable_id(_tr, ["player_id", "transfer_date", "from_club_id", "to_club_id"])
T["transfers"] = _tr[["transfer_id"] + [c for c in _tr.columns if c != "transfer_id"]]

print()
for k in ["competitions", "clubs", "players", "games", "appearances",
          "game_events", "player_valuations", "transfers"]:
    print(f"  {k:<20} {len(T[k]):>10,} rows  {len(T[k].columns):>3} cols")

---
## 6. D-16 — scoped orphan audit, then repair

Stage 2 measured orphans on **full** tables. Those are the wrong numbers to design against:
scoping changes them in both directions. Scoped `games` should be clean by construction
(both clubs played in a covered competition), while `transfers` stays dirty by nature — a
move *out of* the Big 5 points at a club that was never in a covered competition and so is
absent from the 796-row `clubs` table entirely.

This is the step that, skipped, kills the `COPY` inside Terraform at Start Lab.

In [ ]:
FK_EDGES = [
    ("clubs",             "domestic_competition_id", "competitions", "competition_id"),
    ("players",           "current_club_id",         "clubs",        "club_id"),
    ("games",             "competition_id",          "competitions", "competition_id"),
    ("games",             "home_club_id",            "clubs",        "club_id"),
    ("games",             "away_club_id",            "clubs",        "club_id"),
    ("appearances",       "game_id",                 "games",        "game_id"),
    ("appearances",       "player_id",               "players",      "player_id"),
    ("appearances",       "player_club_id",          "clubs",        "club_id"),
    ("appearances",       "competition_id",          "competitions", "competition_id"),
    ("game_events",       "game_id",                 "games",        "game_id"),
    ("game_events",       "player_id",               "players",      "player_id"),
    ("game_events",       "club_id",                 "clubs",        "club_id"),
    ("player_valuations", "player_id",               "players",      "player_id"),
    ("player_valuations", "current_club_id",         "clubs",        "club_id"),
    ("transfers",         "player_id",               "players",      "player_id"),
    ("transfers",         "from_club_id",            "clubs",        "club_id"),
    ("transfers",         "to_club_id",              "clubs",        "club_id"),
]

def audit(label):
    print(f"\n{label}")
    print(f"  {'child.column':<42}{'orphans':>12}{'of':>12}{'pct':>9}")
    print("  " + "-" * 75)
    rows = []
    for ct, cc, pt, pc in FK_EDGES:
        child, parent = T[ct], T[pt]
        vals = child[cc].dropna()
        orph = (~vals.isin(set(parent[pc]))).sum()
        pct = orph / len(child) if len(child) else 0
        flag = "  <-- REPAIR" if orph else ""
        print(f"  {ct+'.'+cc:<42}{orph:>12,}{len(child):>12,}{pct:>8.2%}{flag}")
        rows.append((ct, cc, int(orph), len(child)))
    return rows

before_rows = audit("BEFORE REPAIR (scoped)")

In [ ]:
# --- S-24: NULL the orphans, keep the column nullable ----------------------
# NOT a synthetic placeholder row (puts a fake club in a table students browse) and NOT
# dropping the constraint (Lab 2's LLM uses declared FKs as relationship hints).
#
# The teaching upside: transfers.to_club_id IS NULL now genuinely means "left the Big 5",
# which makes a real LEFT JOIN / NULL-semantics moment in Lab 2 instead of a wart.

NEVER_NULL = {("appearances", "game_id"), ("game_events", "game_id"),
              ("appearances", "player_id"), ("player_valuations", "player_id"),
              ("transfers", "player_id"), ("games", "competition_id")}

repairs = {}
for ct, cc, pt, pc in FK_EDGES:
    child, parent = T[ct], T[pt]
    ok = set(parent[pc])
    mask = child[cc].notna() & ~child[cc].isin(ok)
    n = int(mask.sum())
    if not n:
        continue
    if (ct, cc) in NEVER_NULL:
        raise RuntimeError(
            f"{ct}.{cc} has {n:,} orphans but is NOT NULL in the schema.\n"
            "This means the scoping rule itself is inconsistent — a child row survived "
            "scoping while its parent did not. Fix the scope, don't NULL this."
        )
    T[ct].loc[mask, cc] = pd.NA
    repairs[f"{ct}.{cc}"] = n

print("REPAIRS APPLIED (S-24)")
if repairs:
    for k, v in sorted(repairs.items(), key=lambda kv: -kv[1]):
        print(f"   {k:<42} {v:>10,} set to NULL")
else:
    print("   none needed")

after_rows = audit("AFTER REPAIR (scoped)")
assert all(o == 0 for _, _, o, _ in after_rows), "orphans remain after repair"
print("\n✅ every declared foreign key is now satisfiable — the COPY will not die at Start Lab")

---
## 7. Validation

Cheap assertions that would each have cost a live event. Run before anything is exported.

In [ ]:
problems = []

def check(cond, msg):
    print(("  ✅ " if cond else "  ❌ ") + msg)
    if not cond:
        problems.append(msg)

print("PRIMARY KEYS")
for t, pk in [("competitions", ["competition_id"]), ("clubs", ["club_id"]),
              ("players", ["player_id"]), ("games", ["game_id"]),
              ("appearances", ["appearance_id"]), ("game_events", ["game_event_id"]),
              ("player_valuations", ["player_id", "valuation_date"]),
              ("transfers", ["transfer_id"])]:
    d = T[t]
    check(not d.duplicated(subset=pk).any() and not d[pk].isna().any().any(),
          f"{t}: {'+'.join(pk)} unique and non-null")

print("\nSCOPE COHERENCE")
check(set(T["appearances"].game_id).issubset(set(T["games"].game_id)),
      "every appearance's game is loaded")
check(set(T["game_events"].game_id).issubset(set(T["games"].game_id)),
      "every event's game is loaded")
check(set(T["appearances"].player_id).issubset(set(T["players"].player_id)),
      "every appearance's player is loaded")

print("\nCONTENT")
check(len(T["players"]) > 5000, f"player corpus is worth embedding ({len(T['players']):,})")
check(T["players"].player_name.notna().all(), "no player is missing a name (BM25 target)")
_l = T["players"][T["players"].player_name.str.strip().str.lower() == "lionel messi"]
check(len(_l) == 1, "Lionel Messi present exactly once (Lab 1 canary)")
_dupes = T["players"].player_name.duplicated().sum()
print(f"  ℹ️  {_dupes:,} players share a name with another player — profile text must "
      f"disambiguate by club and era (P-30)")

print("\nPOST-HANDBACK FIXES")
_el = T["competitions"]
_uefa = _el[_el.competition_id.isin(UEFA_COMPS)]
check(len(_uefa) and (_uefa.competition_type == "international_cup").all(),
      "every in-scope UEFA competition is typed international_cup")
_tr = T["transfers"]
_orphan_named = (_tr.from_club_id.isna() & _tr.from_club_name.notna()).sum()
check("from_club_name" in _tr.columns,
      f"transfers keeps club names ({_orphan_named:,} rows have a name but no id)")
check("international_caps" in T["players"].columns,
      f"players carries international_caps "
      f"({T['players'].international_caps.notna().mean():.0%} populated in scope)")

print("\nTYPES")
check(str(T["clubs"].net_transfer_record_eur.dtype).startswith(("int", "float", "Int")),
      f"clubs.net_transfer_record_eur is numeric, not '+€5.90m' ({T['clubs'].net_transfer_record_eur.dtype})")

if problems:
    raise AssertionError(f"{len(problems)} validation failure(s): {problems}")
print("\n✅ all validations passed")

---
## 8. Emit `schema.sql`

The DDL lives here, in the notebook, so it can never drift from the data that was just
built. The loader consumes this file; the schema document describes it.

Two things worth noticing in the DDL:

- **Every club-pointing FK is nullable**, because S-24 NULLs orphans rather than inventing a
  placeholder club. `NOT NULL` appears only where scoping guarantees a parent exists.
- **`COMMENT ON` is not decoration.** Lab 2 points an LLM at this schema, and QueryData
  context sets read column comments. Each one is a hint that buys text-to-SQL accuracy —
  especially the ones that warn about NULL semantics and duplicate names.

In [ ]:
DDL = r"""
-- ===========================================================================
--  CymbalGoal — schema for PostgreSQL 18 on AlloyDB
--  Generated by cymbalgoal_stage3_schema.ipynb. Do not hand-edit.
--  Snapshot: {snapshot_date}  ({snapshot_sha_short})
--  Scope:    {scope}
-- ===========================================================================
-- Extensions are created by the startup VM; listed for completeness.
--   CREATE EXTENSION IF NOT EXISTS vector;
--   CREATE EXTENSION IF NOT EXISTS alloydb_scann;
--   CREATE EXTENSION IF NOT EXISTS google_ml_integration;
--   CREATE EXTENSION IF NOT EXISTS pg_textsearch;   -- BM25, built in Lab 1 Task 3

-- ---------------------------------------------------------------------------
CREATE TABLE competitions (
    competition_id          TEXT        PRIMARY KEY,
    competition_code        TEXT,
    competition_name        TEXT        NOT NULL,
    competition_type        TEXT,
    competition_sub_type    TEXT,
    country_name            TEXT,
    domestic_league_code    TEXT,
    confederation           TEXT,
    total_clubs             INTEGER,
    url                     TEXT
);
COMMENT ON TABLE  competitions IS
    'One row per football competition: domestic leagues, domestic cups, and international club tournaments. Loaded in full, unfiltered.';
COMMENT ON COLUMN competitions.competition_id IS
    'Transfermarkt competition code. The Big 5 domestic leagues are GB1 (Premier League), ES1 (LaLiga), IT1 (Serie A), L1 (Bundesliga), FR1 (Ligue 1).';
COMMENT ON COLUMN competitions.competition_type IS
    'One of: domestic_league, domestic_cup, international_cup, national_team_competition, other. NOTE: normalized during data preparation. The source is self-inconsistent about UEFA competitions - it types the Champions League as international_cup but the Europa League as ''other'' - which would make a filter on international_cup silently return incomplete results. All UEFA club competitions, qualifying rounds included, are normalized to international_cup. Be aware that several competitions in this table have no matches in the games table, because only the Big 5 leagues plus the Champions and Europa Leagues are in scope.';
COMMENT ON COLUMN competitions.country_name IS
    'NULL for international competitions, which belong to no single country.';

-- ---------------------------------------------------------------------------
CREATE TABLE clubs (
    club_id                     INTEGER     PRIMARY KEY,
    club_code                   TEXT,
    club_name                   TEXT        NOT NULL,
    domestic_competition_id     TEXT        REFERENCES competitions (competition_id),
    squad_size                  INTEGER,
    average_age                 NUMERIC(4,1),
    foreigners_number           INTEGER,
    foreigners_percentage       NUMERIC(5,2),
    national_team_players       INTEGER,
    stadium_name                TEXT,
    stadium_seats               INTEGER,
    net_transfer_record_eur     BIGINT,
    coach_name                  TEXT,
    last_season                 INTEGER,
    url                         TEXT,
    profile_text                TEXT,
    profile_embedding           VECTOR({dims})
);
COMMENT ON TABLE  clubs IS
    'One row per football club. Loaded in full, unfiltered, so that fixtures and transfers involving clubs outside the covered competitions still resolve where possible.';
COMMENT ON COLUMN clubs.net_transfer_record_eur IS
    'Net transfer spend in whole EUR. Negative means the club sold more than it bought. Parsed from a formatted source string; safe to SUM and AVG.';
COMMENT ON COLUMN clubs.last_season IS
    'Starting year of the most recent season this club appears in. The 2024/25 season is 2024.';
COMMENT ON COLUMN clubs.profile_text IS
    'Generated free-text club description. Target of the BM25 full-text index.';
COMMENT ON COLUMN clubs.profile_embedding IS
    'Vector embedding of profile_text, {dims} dimensions, model {model}. NULL until the profile load pass runs.';

-- ---------------------------------------------------------------------------
CREATE TABLE players (
    player_id                       INTEGER     PRIMARY KEY,
    player_name                     TEXT        NOT NULL,
    first_name                      TEXT,
    last_name                       TEXT,
    current_club_id                 INTEGER     REFERENCES clubs (club_id),
    country_of_citizenship          TEXT,
    country_of_birth                TEXT,
    city_of_birth                   TEXT,
    date_of_birth                   DATE,
    main_position                   TEXT,
    detailed_position               TEXT,
    foot                            TEXT,
    height_in_cm                    INTEGER,
    market_value_in_eur             BIGINT,
    highest_market_value_in_eur     BIGINT,
    contract_expiration_date        DATE,
    agent_name                      TEXT,
    international_caps              INTEGER,
    international_goals             INTEGER,
    last_season                     INTEGER,
    url                             TEXT,
    profile_text                    TEXT,
    profile_embedding               VECTOR({dims})
);
COMMENT ON TABLE  players IS
    'One row per player who has appeared at least once in a covered competition. The central entity for search: profile_text drives full-text and semantic queries.';
COMMENT ON COLUMN players.player_name IS
    'Full common name, e.g. "Lionel Messi". This is the column exact-name searches target. WARNING: names are NOT unique - many players share a name (e.g. several players named Paulinho). Disambiguate using current_club_id, date_of_birth, or last_season.';
COMMENT ON COLUMN players.current_club_id IS
    'The player''s club as of the snapshot. NULL when that club is not in the loaded club set - a league CymbalGoal does not cover, a youth or reserve side, or no club at all. Note that active, famous players can have a NULL here: anyone who has moved outside the covered leagues.';
COMMENT ON COLUMN players.main_position IS
    'Broad position: Goalkeeper, Defender, Midfield, or Attack.';
COMMENT ON COLUMN players.detailed_position IS
    'Specific role, e.g. Centre-Forward, Left Winger, Defensive Midfield.';
COMMENT ON COLUMN players.market_value_in_eur IS
    'Current Transfermarkt market valuation in whole EUR. See player_valuations for history.';
COMMENT ON COLUMN players.international_caps IS
    'Senior national-team appearances. NULL means unknown, NOT zero - only about 45% of players here have a recorded value, so a majority are NULL. Do not COALESCE to 0 when averaging or ranking; filter with IS NOT NULL instead.';
COMMENT ON COLUMN players.international_goals IS
    'Senior national-team goals. Same NULL caveat as international_caps.';
COMMENT ON COLUMN players.profile_embedding IS
    'Vector embedding of profile_text, {dims} dimensions, model {model}. Always filter WHERE profile_embedding IS NOT NULL in vector queries.';

-- ---------------------------------------------------------------------------
CREATE TABLE games (
    game_id                 INTEGER     PRIMARY KEY,
    competition_id          TEXT        NOT NULL REFERENCES competitions (competition_id),
    season                  INTEGER     NOT NULL,
    round_label             TEXT,
    game_date               DATE        NOT NULL,
    home_club_id            INTEGER     REFERENCES clubs (club_id),
    away_club_id            INTEGER     REFERENCES clubs (club_id),
    home_club_goals         INTEGER,
    away_club_goals         INTEGER,
    home_club_position      INTEGER,
    away_club_position      INTEGER,
    home_club_manager_name  TEXT,
    away_club_manager_name  TEXT,
    home_club_formation     TEXT,
    away_club_formation     TEXT,
    aggregate               TEXT,
    stadium                 TEXT,
    attendance              INTEGER,
    referee                 TEXT,
    url                     TEXT
);
COMMENT ON TABLE  games IS
    'One row per match in a covered competition. Club names come from the clubs table via home_club_id and away_club_id - this table intentionally stores no club name columns.';
COMMENT ON COLUMN games.season IS
    'Starting year of the season. The 2024/25 season is 2024. Note that matches per season vary by year: leagues have changed size, and the 2019 season was truncated.';
COMMENT ON COLUMN games.round_label IS
    'Free text describing the matchday or knockout round, e.g. "8. Matchday", "Round of 16".';
COMMENT ON COLUMN games.aggregate IS
    'Final score as display text, e.g. "2:1". Use home_club_goals and away_club_goals for any arithmetic.';

-- ---------------------------------------------------------------------------
CREATE TABLE appearances (
    appearance_id       TEXT        PRIMARY KEY,
    game_id             INTEGER     NOT NULL REFERENCES games (game_id),
    player_id           INTEGER     NOT NULL REFERENCES players (player_id),
    player_club_id      INTEGER     REFERENCES clubs (club_id),
    competition_id      TEXT        REFERENCES competitions (competition_id),
    appearance_date     DATE,
    yellow_cards        SMALLINT    NOT NULL DEFAULT 0,
    red_cards           SMALLINT    NOT NULL DEFAULT 0,
    goals               SMALLINT    NOT NULL DEFAULT 0,
    assists             SMALLINT    NOT NULL DEFAULT 0,
    minutes_played      SMALLINT    NOT NULL DEFAULT 0
);
COMMENT ON TABLE  appearances IS
    'One row per player per game played. The fact table for all player statistics: goals, assists, cards, minutes. Join to players for names - this table intentionally stores none.';
COMMENT ON COLUMN appearances.player_club_id IS
    'The club the player represented in THIS game, which may differ from players.current_club_id if they later transferred. Use this column for historical questions.';

-- ---------------------------------------------------------------------------
CREATE TABLE game_events (
    game_event_id       {event_pk_type}        PRIMARY KEY,
    game_id             INTEGER     NOT NULL REFERENCES games (game_id),
    player_id           INTEGER     REFERENCES players (player_id),
    club_id             INTEGER     REFERENCES clubs (club_id),
    event_date          DATE,
    event_type          TEXT        NOT NULL,
    minute              SMALLINT,
    description         TEXT,
    player_in_id        INTEGER,
    player_assist_id    INTEGER
);
COMMENT ON TABLE  game_events IS
    'One row per in-match event with a minute marker. event_type is Goals, Cards, Substitutions, or Shootout.';
COMMENT ON COLUMN game_events.game_event_id IS
    'Event primary key. Source: {event_pk_source}.';
COMMENT ON COLUMN game_events.player_id IS
    'NULL when the event references a player outside the loaded player set.';
COMMENT ON COLUMN game_events.player_in_id IS
    'For substitutions, the player coming on. Deliberately NOT a foreign key: the incoming player may fall outside the loaded scope.';
COMMENT ON COLUMN game_events.player_assist_id IS
    'For goals, the assisting player. Deliberately NOT a foreign key, for the same reason.';

-- ---------------------------------------------------------------------------
CREATE TABLE player_valuations (
    player_id           INTEGER     NOT NULL REFERENCES players (player_id),
    valuation_date      DATE        NOT NULL,
    market_value_in_eur BIGINT,
    current_club_id     INTEGER     REFERENCES clubs (club_id),
    PRIMARY KEY (player_id, valuation_date)
);
COMMENT ON TABLE  player_valuations IS
    'Market value history: one row each time a player valuation changed. Designed for window functions over time - LAG, LEAD, and running deltas.';
COMMENT ON COLUMN player_valuations.current_club_id IS
    'The player''s club at valuation time. NULL when that club is not in the loaded club set (about 8% of rows) - same situations as players.current_club_id.';

-- ---------------------------------------------------------------------------
CREATE TABLE transfers (
    transfer_id             BIGINT      PRIMARY KEY,
    player_id               INTEGER     NOT NULL REFERENCES players (player_id),
    transfer_date           DATE        NOT NULL,
    transfer_season         TEXT,
    from_club_id            INTEGER     REFERENCES clubs (club_id),
    from_club_name          TEXT,
    to_club_id              INTEGER     REFERENCES clubs (club_id),
    to_club_name            TEXT,
    transfer_fee            BIGINT,
    market_value_in_eur     BIGINT
);
COMMENT ON TABLE  transfers IS
    'One row per completed transfer. Usually club-to-club, but not always: a row may record a move from free agency (from_club_name = ''Without Club'') or from a youth or reserve side, so do not assume both ends are clubs in the clubs table.';
COMMENT ON COLUMN transfers.transfer_id IS
    'Surrogate key generated deterministically during data preparation; the source has no natural primary key.';
COMMENT ON COLUMN transfers.from_club_id IS
    'Selling club, or NULL when that club is not in the loaded club set. NULL covers three distinct situations and from_club_name tells them apart: (1) a club in a league CymbalGoal does not cover, (2) a youth or reserve side such as RM Castilla or Ajax U21, (3) a non-club state such as ''Without Club'' (free agent) or ''Unknown''. Always LEFT JOIN, and read from_club_name before drawing any conclusion from a NULL.';
COMMENT ON COLUMN transfers.to_club_id IS
    'Buying club, or NULL when that club is not in the loaded club set. Same three NULL situations as from_club_id, and to_club_name likewise distinguishes them.';
COMMENT ON COLUMN transfers.from_club_name IS
    'Name of the selling club, always populated, retained deliberately even when from_club_id is NULL. This is an intentional exception to the rule that name columns are dropped where a foreign key exists: from_club_id is NULL for roughly half these rows, so the name is the ONLY surviving record of where the player came from. Use it for display and for filtering, and from_club_id for joins. Useful values: ''Without Club'' identifies a free-agent signing (expect transfer_fee of 0), and ''Unknown'' means the source itself did not know.';
COMMENT ON COLUMN transfers.to_club_name IS
    'Name of the buying club, always populated, retained even when to_club_id is NULL. Same rationale and same special values as from_club_name.';
COMMENT ON COLUMN transfers.transfer_fee IS
    'Fee in whole EUR. NULL means the fee is unknown; 0 means a free transfer. The distinction is meaningful - do not COALESCE NULL to zero.';
""".format(
    snapshot_date=SNAPSHOT_DATE,
    snapshot_sha_short=SNAPSHOT_SHA256[:16],
    scope=", ".join(SCOPE_COMPS),
    dims=EMBEDDING_DIMS,
    model=EMBEDDING_MODEL,
    event_pk_type=EVENT_PK_TYPE,
    event_pk_source=EVENT_PK_SOURCE,
)

INDEXES = r"""
-- ===========================================================================
--  Indexes — built by the startup VM AFTER the load, never staged.
-- ===========================================================================
-- PostgreSQL does not index foreign key columns automatically.
CREATE INDEX idx_appearances_player       ON appearances (player_id);
CREATE INDEX idx_appearances_game         ON appearances (game_id);
CREATE INDEX idx_game_events_game         ON game_events (game_id);
CREATE INDEX idx_player_valuations_player ON player_valuations (player_id);
CREATE INDEX idx_transfers_player         ON transfers (player_id);
CREATE INDEX idx_games_competition_season ON games (competition_id, season);

-- Vector search. ScaNN at 3072 dimensions is proven in the shipped CymbalFlix lab.
CREATE INDEX players_profile_embedding_scann_idx
    ON players USING scann (profile_embedding cosine)
    WITH (num_leaves = {player_leaves}, quantizer = 'sq8');

CREATE INDEX clubs_profile_embedding_scann_idx
    ON clubs USING scann (profile_embedding cosine)
    WITH (num_leaves = {club_leaves}, quantizer = 'sq8');

-- NOTE: the BM25 index is deliberately NOT created here. Building it is Lab 1 Task 3 —
-- students need to watch keyword search fail before they add it.
""".format(
    player_leaves=max(10, int(len(T["players"]) ** 0.5)),
    club_leaves=max(5, int(len(T["clubs"]) ** 0.5)),
)

ddl_path = ARTIFACT / "schema.sql"
ddl_path.write_text(DDL + "\n" + INDEXES)
print(f"✅ wrote {ddl_path}  ({len(DDL)+len(INDEXES):,} bytes)")
print(f"   ScaNN num_leaves: players={max(10, int(len(T['players'])**0.5))}, "
      f"clubs={max(5, int(len(T['clubs'])**0.5))}  (sqrt of row count; ScaNN self-tunes from there)")


# --- Align every table to DDL column order -----------------------------------
# The transform builds columns in whatever order the rename map plus keep_extra
# produces, which is NOT necessarily DDL order. Left alone, the staged CSV and the
# CREATE TABLE disagree, and a positional COPY loads the wrong column into the wrong
# field. Deriving the order from the DDL makes divergence impossible rather than
# merely unlikely.
def ddl_column_order(ddl):
    order, cur = {}, None
    for line in ddl.splitlines():
        m = re.match(r"\s*CREATE TABLE (\w+)", line)
        if m:
            cur = m.group(1); order[cur] = []; continue
        if cur is None:
            continue
        if line.strip().startswith(")"):
            cur = None; continue
        m = re.match(r"\s+(\w+)\s+[A-Z]", line)
        if m and m.group(1).upper() not in ("PRIMARY", "FOREIGN", "CONSTRAINT", "UNIQUE"):
            order[cur].append(m.group(1))
    return order


DDL_ORDER = ddl_column_order(DDL)
_reordered = []
for t in T:
    want = [c for c in DDL_ORDER[t] if c in T[t].columns]
    extra = [c for c in T[t].columns if c not in DDL_ORDER[t]]
    if extra:
        raise RuntimeError(f"{t} has columns absent from the DDL: {extra}")
    missing = [c for c in DDL_ORDER[t] if c not in T[t].columns and c != "profile_embedding"]
    if missing:
        raise RuntimeError(f"{t} is missing DDL columns: {missing}")
    before = list(T[t].columns)
    if want != before:
        moved = [f"{c}: {before.index(c)} -> {i}" for i, c in enumerate(want)
                 if before.index(c) != i]
        _reordered.append(f"{t}  [{', '.join(moved)}]")
    T[t] = T[t][want]

print(f"\n✅ column order aligned to DDL for all {len(T)} tables")
if _reordered:
    print("   REORDERED (these would have broken a positional COPY):")
    for r in _reordered:
        print(f"     {r}")
else:
    print("   no changes needed")

---
## 9. Checkpoint to Parquet

Parquet is the internal working format — typed, compact, and it round-trips NULLs correctly.
The gzipped-CSV export that AlloyDB actually imports happens in Stage 6, after profiles and
embeddings exist, so that the embedding column is written exactly once.

In [ ]:
LOAD_ORDER = ["competitions", "clubs", "players", "games",
              "appearances", "game_events", "player_valuations", "transfers"]

# Parent before child. Any other order fails on foreign keys.
DEPENDS = {
    "competitions":      [],
    "clubs":             ["competitions"],
    "players":           ["clubs"],
    "games":             ["competitions", "clubs"],
    "appearances":       ["games", "players", "clubs", "competitions"],
    "game_events":       ["games", "players", "clubs"],
    "player_valuations": ["players", "clubs"],
    "transfers":         ["players", "clubs"],
}
for i, t in enumerate(LOAD_ORDER):
    for dep in DEPENDS[t]:
        assert LOAD_ORDER.index(dep) < i, f"load order violates {t} -> {dep}"
print("✅ load order respects every parent-before-child dependency\n")

written = {}
for t in LOAD_ORDER:
    p = PARQUET / f"{t}.parquet"
    T[t].to_parquet(p, index=False, compression="snappy")
    written[t] = p.stat().st_size
    print(f"  {t:<20} {len(T[t]):>10,} rows   {written[t]/1e6:>8.1f} MB")

print(f"\ntotal {sum(written.values())/1e6:.1f} MB "
      f"(embedding columns not yet populated — Stage 5 adds ~0.13 GB gzipped)")


---
## 9a. Stage 6a — export relational CSVs and stage to GCS

**Why this runs now rather than in the profile session.** None of these eight tables have a
relational column that Stage 4 or 5 touches. `profile_text` and `profile_embedding` load in a
**second pass**, keyed on `player_id` / `club_id`, so the relational extract is final today.
Staging it now lets the Terraform session build and test the loader against real files instead of
waiting on 14,235 grounded generations.

Three things this has to get right, all of which fail *at import time* rather than here:

- **Integer columns that took NULLs became floats.** pandas widens `int64` to `float64` the moment
  a NaN appears, so `current_club_id` is `69261.0` in memory. PostgreSQL `INTEGER` rejects
  `69261.0`. Every integer column is cast to pandas nullable `Int64` before writing, with the type
  list **parsed from the DDL we just generated** so the two cannot drift apart.
- **No header row.** An AlloyDB CSV import requirement.
- **NULL is an unquoted empty field.** That is PostgreSQL's CSV default, and it is why `na_rep=""`
  with `QUOTE_MINIMAL` is correct — a quoted `""` would load as an empty string, not NULL.


In [ ]:

import csv, gzip, subprocess

EXPORT_DIR = BASE / "csv"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

PASS2_COLS = {"profile_text", "profile_embedding"}
INT_TYPES  = {"INTEGER", "BIGINT", "SMALLINT"}


def parse_ddl_types(ddl):
    """Column -> SQL type, read straight from the DDL so the two cannot drift."""
    types, cur = {}, None
    for line in ddl.splitlines():
        m = re.match(r"\s*CREATE TABLE (\w+)", line)
        if m:
            cur = m.group(1); types[cur] = {}; continue
        if cur is None:
            continue
        if line.strip().startswith(")"):
            cur = None; continue
        m = re.match(r"\s+(\w+)\s+([A-Z]+(?:\(\d+(?:,\d+)?\))?)", line)
        if m and m.group(1).upper() not in ("PRIMARY", "FOREIGN", "CONSTRAINT", "REFERENCES", "UNIQUE"):
            types[cur][m.group(1)] = m.group(2)
    return types


COLTYPES = parse_ddl_types(DDL)


def prep_for_csv(df, tbl):
    """Coerce to something PostgreSQL will actually accept."""
    out, fixed = df.copy(), []
    for c in out.columns:
        t = COLTYPES.get(tbl, {}).get(c, "")
        if t in INT_TYPES:
            before = str(out[c].dtype)
            out[c] = pd.to_numeric(out[c], errors="coerce").astype("Int64")
            if before.startswith("float"):
                fixed.append(c)
        elif t == "DATE":
            out[c] = pd.to_datetime(out[c], errors="coerce").dt.strftime("%Y-%m-%d")
    return out, fixed


staged_manifest = {}
print(f"{'table':<20}{'rows':>10}{'cols':>6}{'gz bytes':>12}  float->Int64 repairs")
print("-" * 78)

for t in LOAD_ORDER:
    cols = [c for c in T[t].columns if c not in PASS2_COLS]
    expect = [c for c in COLTYPES[t] if c not in PASS2_COLS]
    if cols != expect:
        raise RuntimeError(
            f"{t}: CSV column order does not match the DDL.\n"
            f"  csv: {cols}\n  ddl: {expect}\n"
            "A positional COPY would load the wrong column into the wrong field."
        )
    df, fixed = prep_for_csv(T[t][cols], t)
    path = EXPORT_DIR / f"{t}.csv.gz"

    df.to_csv(path, index=False, header=False, na_rep="",
              quoting=csv.QUOTE_MINIMAL, encoding="utf-8",
              compression="gzip", lineterminator="\n")

    h = hashlib.sha256(path.read_bytes()).hexdigest()
    staged_manifest[t] = {
        "file": f"{t}.csv.gz",
        "rows": int(len(df)),
        "column_order": list(df.columns),
        "gz_bytes": path.stat().st_size,
        "sha256": h,
    }
    print(f"{t:<20}{len(df):>10,}{len(df.columns):>6}{path.stat().st_size:>12,}  "
          f"{', '.join(fixed) if fixed else '-'}")

print(f"\ntotal staged: {sum(v['gz_bytes'] for v in staged_manifest.values())/1e6:.1f} MB")
print(f"NOTE: profile_text / profile_embedding deliberately excluded — they load in pass 2.")


In [ ]:

# --- read every file back before anything touches the bucket -----------------
problems = []
for t in LOAD_ORDER:
    path = EXPORT_DIR / f"{t}.csv.gz"
    expect_cols = len(staged_manifest[t]["column_order"])
    n, bad, floaty = 0, 0, 0
    int_idx = [i for i, c in enumerate(staged_manifest[t]["column_order"])
               if COLTYPES.get(t, {}).get(c, "") in INT_TYPES]
    with gzip.open(path, "rt", encoding="utf-8", newline="") as fh:
        for row in csv.reader(fh):
            n += 1
            if len(row) != expect_cols:
                bad += 1
            if bad == 0 and n <= 5000:
                for i in int_idx:
                    if row[i] and ("." in row[i] or "e" in row[i].lower()):
                        floaty += 1
    ok_rows = (n == staged_manifest[t]["rows"])
    status = "ok"
    if not ok_rows:
        status = f"ROW COUNT {n:,} != {staged_manifest[t]['rows']:,}"; problems.append(t)
    elif bad:
        status = f"{bad:,} rows with wrong field count"; problems.append(t)
    elif floaty:
        status = f"{floaty:,} float-formatted integers"; problems.append(t)
    print(f"  {'✅' if status=='ok' else '❌'} {t:<20}{n:>10,} rows x {expect_cols} cols   {status}")

if problems:
    raise AssertionError(f"export validation failed: {problems}")
print("\n✅ every file re-reads cleanly: row counts match, field counts uniform, "
      "no float-formatted integers")


In [ ]:

# --- upload. gcloud storage, never gsutil ------------------------------------
dest = GCS_PREFIX.rstrip("/") + "/"
files = sorted(EXPORT_DIR.glob("*.csv.gz")) + [ARTIFACT / "schema.sql"]

cmd = ["gcloud", "storage", "cp", *[str(f) for f in files], dest]
print(" ".join(cmd[:4]) + f" … ({len(files)} files) {dest}\n")
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout or "", r.stderr or "")
if r.returncode != 0:
    raise RuntimeError(
        f"upload failed (exit {r.returncode}).\n"
        "Check: does the bucket exist, and does this identity have storage.objects.create on it?\n"
        "  gcloud storage ls gs://class-demo/"
    )

print("--- what is actually in the bucket now ---")
ls = subprocess.run(["gcloud", "storage", "ls", "-l", dest],
                    capture_output=True, text=True)
print(ls.stdout or ls.stderr)
UPLOADED = True


---
## 10. Manifest

Lab verification steps assert against these values, and the Terraform session reads
`column_order` to write an explicit `\copy` column list. Positional loading is how a schema
change six weeks from now silently shifts every column by one.

In [ ]:
manifest = {
    "generated_at_utc":   datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "generator":          "cymbalgoal_stage3_schema.ipynb",
    "stage":              "3 — schema build (profiles and embeddings not yet generated)",

    "snapshot": {
        "date":   SNAPSHOT_DATE,
        "sha256": SNAPSHOT_SHA256,
        "bytes":  SNAPSHOT_BYTES,
        "url":    SNAPSHOT_URL,
    },
    "scope": {
        "competitions":       SCOPE_COMPS,
        "big5":               BIG5,
        "uefa_discovered":    UEFA_COMPS,
        "player_rule":        "S-23 option A — ever appeared in a scope competition, any season",
        "dimension_rule":     "S-21 — competitions and clubs loaded whole, facts scoped",
    },
    "embedding": {
        "model":               EMBEDDING_MODEL,
        "dimensions":          EMBEDDING_DIMS,
        "significant_digits":  VECTOR_SIGFIGS,
        "locked_reason":       "P-25 — AlloyDB's in-SQL embedding() takes no output_dimensionality; "
                               "query-time vectors are 3072 so storage must match",
    },
    "destination":  GCS_PREFIX,
    "staged_to_gcs": bool(globals().get("UPLOADED")),
    "staged_files":  globals().get("staged_manifest", {}),
    "load_order":   LOAD_ORDER,
    "fk_repairs":   repairs,
    "event_pk_source": EVENT_PK_SOURCE,
    "profile_columns_are_pass_2": ["profile_text", "profile_embedding"],

    "tables": {
        t: {
            "rows":          int(len(T[t])),
            "column_order":  list(T[t].columns),
            "parquet_bytes": int(written[t]),
        } for t in LOAD_ORDER
    },
}

mpath = ARTIFACT / "manifest.json"
mpath.write_text(json.dumps(manifest, indent=2))
print(json.dumps({k: v for k, v in manifest.items() if k != "tables"}, indent=2))
print(f"\n✅ wrote {mpath}")

---
## 11. Handoff summary

In [ ]:
print("=" * 78)
print("CYMBALGOAL STAGE 3 — COMPLETE")
print("=" * 78)
print(f"\nsnapshot   {SNAPSHOT_DATE}  {SNAPSHOT_SHA256[:16]}…")
print(f"scope      {len(SCOPE_COMPS)} competitions: {', '.join(SCOPE_COMPS)}")
print(f"players    {len(T['players']):,}  (option A — ever appeared in scope)")
print(f"clubs      {len(T['clubs']):,}  (loaded whole)")

print("\nROW COUNTS")
for t in LOAD_ORDER:
    print(f"   {t:<20} {len(T[t]):>10,}")

print("\nFK REPAIRS (S-24)")
if repairs:
    for k, v in sorted(repairs.items(), key=lambda kv: -kv[1]):
        print(f"   {k:<42} {v:>10,} -> NULL")
else:
    print("   none needed")

print("\nARTIFACTS")
for p in sorted(ARTIFACT.glob("*")):
    print(f"   {p}  ({p.stat().st_size:,} bytes)")
print(f"   {PARQUET}/*.parquet  ({sum(written.values())/1e6:.1f} MB total)")

print("\nSTAGED TO GCS")
if globals().get("UPLOADED"):
    for t, v in staged_manifest.items():
        print(f"   {v['file']:<28}{v['rows']:>10,} rows{v['gz_bytes']/1e6:>9.1f} MB")
    print(f"   -> {GCS_PREFIX}/")
    print("   (relational columns only; profile_text + profile_embedding are pass 2)")
else:
    print("   NOT UPLOADED — Stage 6a did not run")

print("\nNEXT")
print("   Stage 4 — generate player and club profile text with Gemini + Search grounding")
print("             temperature 0 (P-27); show samples before the full run")
print("   Stage 5 — embed profile_text at 3072 dims, 6 significant digits")
print("   Stage 6b — stage ONLY the pass-2 profile file; relational data is already up")
print("=" * 78)